In [1]:
from transformer_block import TransformerBlock
from tokenization_and_embedding import TokenAndPositionEmbedding
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import umap
import sentencepiece as spm

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split

2026-03-05 09:41:54.980707: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 09:41:55.221469: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 09:41:57.437129: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
SENTENCE_VECTORS = 8000

df = pd.read_parquet("data.parquet")

X = df["text"].astype(str)
y = df["label"].astype(int)

emotion_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'} 
df['emotion'] = df['label'].map(emotion_map)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

bpe_model = tf.keras.models.load_model(
    "emotion_transformer.keras",
    custom_objects={
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding, 
        "TransformerBlock": TransformerBlock
    }
)

embedding_layer = next(l for l in bpe_model.layers if isinstance(l, TokenAndPositionEmbedding))
embeddings = embedding_layer.get_weights()[0]
# print(embeddings.shape)

feature_model = tf.keras.Model(
    inputs=bpe_model.input,
    outputs=bpe_model.layers[-3].output
)

def encode_for_transformer(sp, texts, max_len=128):
    input_ids = []
    masks = []

    pad_id = sp.pad_id()

    for text in texts:
        ids = sp.encode(text.strip(), out_type=int)[:max_len]
        mask = [1]*len(ids)

        while len(ids) < max_len:
            ids.append(pad_id)
            mask.append(0)

        input_ids.append(ids)
        masks.append(mask)

    return (
        np.array(input_ids, dtype=np.int32),
        np.array(masks, dtype=np.int32)
    )

sp_bpe = spm.SentencePieceProcessor()
sp_bpe.load("m_bpe.model")

X_test_ids, X_test_mask = encode_for_transformer(
    sp_bpe,
    X_test.tolist()
)

test_data = tf.data.Dataset.from_tensor_slices(
    ((X_test_ids, X_test_mask), y_test.values)
).batch(128)

sentence_vectors = feature_model.predict(test_data)
print(sentence_vectors.shape)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(sentence_vectors)

plt.scatter(X_pca[:,0], X_pca[:,1], c=y_test, alpha=0.4)

tsne = TSNE(n_components=3, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(sentence_vectors[:SENTENCE_VECTORS])

reducer = umap.UMAP(n_components=3, min_dist=0.1)
X_umap = reducer.fit_transform(sentence_vectors[:SENTENCE_VECTORS])

lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(sentence_vectors, y_test)

feature_model = tf.keras.Model(
    inputs=bpe_model.input,
    outputs=bpe_model.layers[-3].output
)

sentence_vectors = feature_model.predict(test_data)
print(sentence_vectors.shape)

I0000 00:00:1772703752.656216   14041 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'AdamW', 'config': {'name': 'adamw', 'learning_rate': 0.0003000000142492354, 'weight_decay': 0.0001, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': {'module': 'keras.losses', 'class_name': 'SparseCategoricalCrossentropy', 'confi

In [ ]:
def save_projection_html(X_2d, y, emotion_map, title, out_html):
    """
    X_2d: (n, 2) array
    y:    (n,) labels (ints)
    """
    df_plot = pd.DataFrame({
        "x": X_2d[:, 0],
        "y": X_2d[:, 1],
        "label": y.astype(int),
        "emotion": [emotion_map[int(i)] for i in y.astype(int)]
    })

    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        color="emotion",
        hover_data=["label", "emotion"],
        title=title
    )

    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html}")
    
y_test_np = y_test.to_numpy()

save_projection_html(
    X_pca,
    y_test_np,
    emotion_map,
    title="PCA of Transformer Sentence Vectors (Test Set)",
    out_html=f"{SENTENCE_VECTORS}_pca_sentence_vectors.html"
)

save_projection_html(
    X_tsne,
    y_test_np[:SENTENCE_VECTORS],
    emotion_map,
    title=f"t-SNE of Transformer Sentence Vectors (First {SENTENCE_VECTORS} Test Samples)",
    out_html=f"{SENTENCE_VECTORS}_tsne_sentence_vectors.html"
)

save_projection_html(
    X_umap[:, :2],
    y_test_np[:SENTENCE_VECTORS],
    emotion_map,
    title=f"UMAP (first 2 dims) of Transformer Sentence Vectors (First {SENTENCE_VECTORS} Test Samples)",
    out_html=f"{SENTENCE_VECTORS}_umap_sentence_vectors.html"
)

save_projection_html(
    X_umap[:, :2],
    y_test_np[:SENTENCE_VECTORS],
    emotion_map,
    title=f"UMAP (first 2 dims) of Transformer Sentence Vectors (First {SENTENCE_VECTORS} Test Samples)",
    out_html=f"{SENTENCE_VECTORS}_umap_sentence_vectors.html"
)

In [ ]:
bigru_model = tf.keras.models.load_model("bigru_model.keras")
bigru_model.summary()

try:
    dense_relu_layer = bigru_model.get_layer(index=-3)
except:
    dense_relu_layer = bigru_model.layers[-3]

bigru_feature_model = tf.keras.Model(
    inputs=bigru_model.input,
    outputs=dense_relu_layer.output
)

bigru_vectors = bigru_feature_model.predict(Xu_test, batch_size=128, verbose=1)
print("BiGRU vectors:", bigru_vectors.shape)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import umap

N = min(SENTENCE_VECTORS, len(y_test_np), sentence_vectors.shape[0], bigru_vectors.shape[0])

reducer = umap.UMAP(n_components=2, min_dist=0.1, random_state=42)

X_umap_tr = reducer.fit_transform(sentence_vectors[:N])
X_umap_gru = reducer.fit_transform(bigru_vectors[:N])

df_tr = pd.DataFrame({
    "x": X_umap_tr[:, 0],
    "y": X_umap_tr[:, 1],
    "emotion": [emotion_map[int(i)] for i in y_test_np[:N].astype(int)],
    "model": "Transformer (BPE)"
})

df_gru = pd.DataFrame({
    "x": X_umap_gru[:, 0],
    "y": X_umap_gru[:, 1],
    "emotion": [emotion_map[int(i)] for i in y_test_np[:N].astype(int)],
    "model": "BiGRU (Unigram)"
})

df_both = pd.concat([df_tr, df_gru], ignore_index=True)

fig = px.scatter(
    df_both,
    x="x", y="y",
    color="emotion",
    facet_col="model",
    title=f"UMAP (2D) Sentence Vectors — Side-by-side (N={N})",
    opacity=0.6
)

fig.update_traces(marker=dict(size=5))
fig.write_html(f"{N}_umap_side_by_side_transformer_vs_bigru.html", include_plotlyjs="cdn")
print("Saved side-by-side UMAP HTML")